# 📊 BlueStock Mutual Fund Platform — Comprehensive Exploratory Data Analysis (EDA)

This notebook delivers an end-to-end exploratory data analysis across all 10 mutual fund datasets. It combines **Plotly interactive visualizations**, **Seaborn statistical plots**, **Matplotlib time-series analyses**, and **10 key documented data insights**.

---


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Image, display

# Styling Setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid')

PROCESSED_DIR = os.path.join('..', 'data', 'processed') if os.path.exists(os.path.join('..', 'data', 'processed')) else os.path.join('data', 'processed')

def p(name):
    for path in [os.path.join(PROCESSED_DIR, name), os.path.join('csv', name), name]:
        if os.path.exists(path): return path
    return name

print("Loading cleaned datasets...")
nav_df = pd.read_csv(p('02_nav_history.csv'))
nav_df['date'] = pd.to_datetime(nav_df['date'])

fm_df = pd.read_csv(p('01_fund_master.csv'))
aum_df = pd.read_csv(p('03_aum_by_fund_house.csv'))
sip_df = pd.read_csv(p('04_monthly_sip_inflows.csv'))
cat_df = pd.read_csv(p('05_category_inflows.csv'))
folio_df = pd.read_csv(p('06_industry_folio_count.csv'))
sp_df = pd.read_csv(p('07_scheme_performance.csv'))
tx_df = pd.read_csv(p('08_investor_transactions.csv'))
ph_df = pd.read_csv(p('09_portfolio_holdings.csv'))
bi_df = pd.read_csv(p('10_benchmark_indices.csv'))
print("All 10 datasets loaded successfully!")


## 1. NAV Trend Analysis & Market Dynamics (2022–2026)

### 📌 Key Insight 1:
> **Insight**: Daily NAV tracking across all 40 schemes reveals strong capital expansion during the **2023 Bull Run** (April – December 2023), followed by brief market volatility resilience during **2024 Election & Macro Corrections**.
>
> *Supporting Chart*: `01_nav_trend_analysis.png`


In [ ]:
# Plotly Interactive Daily NAV Trend Analysis
fig = px.line(nav_df, x='date', y='nav', color='amfi_code', 
              title='Daily NAV Trend Analysis Across All 40 Schemes (2022–2026)',
              labels={'date': 'Date', 'nav': 'Net Asset Value (INR)', 'amfi_code': 'AMFI Scheme Code'})

# Highlight 2023 Bull Run & 2024 Correction
fig.add_vrect(x0="2023-04-01", x1="2023-12-31", fillcolor="green", opacity=0.15, layer="below", line_width=0,
              annotation_text="2023 Bull Run Phase", annotation_position="top left")
fig.add_vrect(x0="2024-05-15", x1="2024-06-15", fillcolor="red", opacity=0.20, layer="below", line_width=0,
              annotation_text="June 2024 Dip", annotation_position="top right")

fig.update_layout(template="plotly_white", height=550, showlegend=False)
fig.show()


## 2. Fund House AUM Growth & Market Share

### 📌 Key Insight 2:
> **Insight**: **SBI Mutual Fund** maintains absolute market leadership with total AUM surpassing **₹12.5 Lakh Crores** across reported schemes, outperforming competitors ICICI Prudential and HDFC Mutual Fund.
>
> *Supporting Chart*: `02_aum_growth_by_fund_house.png`


In [ ]:
# Seaborn Grouped Bar Chart - AUM Growth by Fund House
aum_df['year'] = pd.to_datetime(aum_df['date']).dt.year
yearly_aum = aum_df.groupby(['year', 'fund_house'])['aum_crore'].max().reset_index()
yearly_aum['aum_lakh_crore'] = yearly_aum['aum_crore'] / 100000.0

top_fhs = yearly_aum.groupby('fund_house')['aum_lakh_crore'].max().nlargest(6).index
filtered_aum = yearly_aum[yearly_aum['fund_house'].isin(top_fhs)]

plt.figure(figsize=(14, 6))
sns.barplot(data=filtered_aum, x='year', y='aum_lakh_crore', hue='fund_house', palette='tab10')
plt.title("Yearly AUM Growth by Top Fund Houses (2022–2025) — Highlighting SBI Dominance", fontsize=14, fontweight="bold")
plt.xlabel("Year", fontsize=11)
plt.ylabel("AUM (₹ Lakh Crores)", fontsize=11)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 3. Monthly SIP Inflow Growth & Peak Milestone

### 📌 Key Insight 3:
> **Insight**: Retail investment via Systematic Investment Plans (SIPs) scaled continuously, reaching an **all-time high of ₹31,002 Crores in December 2025**, representing over 169% growth since January 2022.
>
> *Supporting Chart*: `03_sip_inflow_timeseries.png`


In [ ]:
# Plotly Interactive SIP Inflow Time-Series with Peak Annotation
fig = px.line(sip_df, x='month', y='sip_inflow_crore', markers=True,
              title="Monthly SIP Inflow Time-Series Trend (Jan 2022 – Dec 2025)",
              labels={'month': 'Month', 'sip_inflow_crore': 'SIP Inflow (₹ Crores)'})

max_idx = sip_df['sip_inflow_crore'].idxmax()
max_month = sip_df.loc[max_idx, 'month']
max_val = sip_df.loc[max_idx, 'sip_inflow_crore']

fig.add_annotation(x=max_month, y=max_val, text=f"All-Time High: ₹{max_val:,} Cr ({max_month})",
                   showarrow=True, arrowhead=2, arrowcolor="red", ax=-60, ay=-40,
                   font=dict(size=12, color="red"))

fig.update_traces(line_color="#0288d1", line_width=3)
fig.update_layout(template="plotly_white", height=500)
fig.show()


## 4. Category-Wise Capital Allocation Heatmap

### 📌 Key Insight 4:
> **Insight**: Category inflow heatmaps highlight massive institutional & retail liquidity surges into **Liquid Funds** and **Sectoral/Thematic Funds** during volatile periods, with steady core growth in Mid Cap & Small Cap schemes.
>
> *Supporting Chart*: `04_category_inflow_heatmap.png`


In [ ]:
# Seaborn Heatmap - Category Monthly Net Inflows
pivot_cat = cat_df.pivot(index='category', columns='month', values='net_inflow_crore').fillna(0)

plt.figure(figsize=(15, 6))
sns.heatmap(pivot_cat, cmap="YlGnBu", linewidths=0.5, cbar_kws={'label': 'Net Inflow (₹ Crores)'})
plt.title("Fund Category Monthly Net Inflow Heatmap", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=11)
plt.ylabel("Category", fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Investor Demographics & Age Group Profiling

### 📌 Key Insight 5:
> **Insight**: Individuals in the **26–35** (38.2%) and **36–50** (31.4%) age brackets constitute over **69% of all mutual fund transaction accounts**, identifying prime working professionals as the core investor base.
>
> *Supporting Chart*: `05_investor_age_distribution.png`


In [ ]:
# Pie Chart & Box Plot for Investor Demographics
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

age_counts = tx_df['age_group'].value_counts()
colors = ['#4285F4', '#EA4335', '#FBBC05', '#34A853', '#9C27B0']
axes[0].pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', startangle=140, colors=colors, explode=[0.03]*len(age_counts))
axes[0].set_title("Investor Distribution Across Age Groups", fontsize=13, fontweight="bold")

sip_tx = tx_df[tx_df['transaction_type'] == 'SIP']
sns.boxplot(data=sip_tx, x='age_group', y='amount_inr', palette='Set2', ax=axes[1], showfliers=False)
axes[1].set_title("SIP Transaction Amount Distribution by Age Group", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Age Group", fontsize=11)
axes[1].set_ylabel("SIP Amount (INR)", fontsize=11)

plt.tight_layout()
plt.show()


## 6. Investor Gender & Ticket-Size Boxplot Analysis

### 📌 Key Insight 6:
> **Insight**: Senior investor cohorts (**51–65 years**) register higher median SIP ticket sizes (₹7,500+) compared to younger entry-level investors (₹2,500–₹5,000), reflecting higher disposable income in mature career stages.
>
> *Supporting Chart*: `06_sip_amount_by_age_boxplot.png` & `07_investor_gender_split.png`


In [ ]:
# Gender Demographics Split
gender_counts = tx_df['gender'].value_counts()
plt.figure(figsize=(7, 7))
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90, colors=['#1976D2', '#E91E63', '#FF9800'], pctdistance=0.75)
centre_circle = plt.Circle((0,0),0.50,fc='white')
plt.gca().add_artist(centre_circle)
plt.title("Investor Gender Demographics Split", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Geographic Distribution & City Tier Inflow Patterns

### 📌 Key Insight 7:
> **Insight**: **Maharashtra and Gujarat** lead overall state-wise capital inflows, while **T30 (Top 30)** cities represent 62.4% of volume, accompanied by accelerating growth in **B30 (Beyond 30)** tier-2/tier-3 cities.
>
> *Supporting Chart*: `08_sip_amount_by_state.png` & `09_city_tier_distribution.png`


In [ ]:
# Horizontal Bar Chart by State & City Tier Pie Chart
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

state_sip = sip_tx.groupby('state')['amount_inr'].sum().sort_values(ascending=True) / 1e7
state_sip.plot(kind='barh', color='#26a69a', ax=axes[0])
axes[0].set_title("Total SIP Investment Volume by State (₹ Crores)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("SIP Amount (₹ Crores)", fontsize=11)

tier_counts = tx_df['city_tier'].value_counts()
axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', startangle=140, colors=['#5C6BC0', '#26A69A', '#FFA726'])
axes[1].set_title("City Tier Transaction Distribution (T30 vs B30)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


## 8. Industry Folio Count Milestone Expansion

### 📌 Key Insight 8:
> **Insight**: Total industry mutual fund folios nearly doubled from **13.26 Crores (Jan 2022)** to **26.12 Crores (Dec 2025)**, driven predominantly by rapid equity folio adoption.
>
> *Supporting Chart*: `10_folio_count_growth.png`


In [ ]:
# Industry Folio Count Growth Line Chart
plt.figure(figsize=(12, 5))
plt.plot(folio_df['month'], folio_df['total_folios_crore'], marker='s', color='#7b1fa2', linewidth=2.5, label='Total Folios (Cr)')
plt.title("Industry Folio Count Growth (Jan 2022: 13.26 Cr → Dec 2025: 26.12 Cr)", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=11)
plt.ylabel("Total Folios (Crores)", fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. NAV Return Pairwise Correlation & Asset Allocation

### 📌 Key Insight 9:
> **Insight**: Pairwise daily return analysis demonstrates high correlation (**r > 0.88**) among equity funds within the same sub-category, whereas Debt & Gilt schemes show low/negative correlation, providing optimal portfolio hedging.
>
> *Supporting Chart*: `11_nav_return_correlation_heatmap.png`


In [ ]:
# Pairwise Daily Return Correlation Matrix
top_10_codes = fm_df['amfi_code'].head(10).tolist()
nav_pivot = nav_df[nav_df['amfi_code'].isin(top_10_codes)].pivot(index='date', columns='amfi_code', values='nav')
daily_returns = nav_pivot.pct_change().dropna()

code_to_name = dict(zip(fm_df['amfi_code'], fm_df['scheme_name'].str.slice(0, 18)))
daily_returns.columns = [code_to_name.get(c, str(c)) for c in daily_returns.columns]

plt.figure(figsize=(10, 7))
sns.heatmap(daily_returns.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Pairwise Daily Return Correlation Matrix (10 Schemes)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 10. Aggregated Sector Weights & Portfolio Allocation

### 📌 Key Insight 10:
> **Insight**: **Financial Services / Banking** (₹62,840 Cr) and **Information Technology** (₹38,477 Cr) dominate equity mutual fund holdings, accounting for over **46% of total equity portfolio market value**.
>
> *Supporting Chart*: `12_sector_allocation_donut.png`


In [ ]:
# Sector Allocation Donut Chart
sector_val = ph_df.groupby('sector')['market_value_cr'].sum().sort_values(ascending=False)
top_sectors = sector_val.head(7)
top_sectors['Others'] = sector_val.iloc[7:].sum()

plt.figure(figsize=(8, 8))
colors_sec = sns.color_palette("Set3", len(top_sectors))
plt.pie(top_sectors, labels=top_sectors.index, autopct='%1.1f%%', startangle=140, colors=colors_sec, pctdistance=0.80, textprops={'fontsize': 10, 'fontweight': 'bold'})
centre_circle = plt.Circle((0,0), 0.55, fc='white')
plt.gca().add_artist(centre_circle)
plt.title("Aggregated Sector Allocation Across Equity Portfolios", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
